# SinLlama 1B — Continual Pre-Training
Runs on Modal GPU workbench. Execute cells top to bottom.

## 1. Install dependencies

In [ ]:
!pip install -q torch transformers datasets peft accelerate sentencepiece scikit-learn huggingface_hub hf_transfer

## 2. HuggingFace login
Paste your token from huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login
login(token="hf_your_token_here")

## 3. Clone your repo

In [ ]:
!git clone https://github.com/isuruij/llama-1B.git
%cd llama-1B/Chinese-LLaMA-Alpaca/scripts/training

## 4. Download the Sinhala dataset (isji/sinhala-corpus)

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import hf_hub_download

os.makedirs('./sinhala_data', exist_ok=True)

print('Downloading sinhala_corpus.txt from isji/sinhala-corpus ...')
hf_hub_download(
    repo_id='isji/sinhala-corpus',
    filename='sinhala_corpus.txt',
    repo_type='dataset',
    local_dir='./sinhala_data',
)
print('Done.')

## 5. Run Continual Pre-Training

In [ ]:
!torchrun \
    --nnodes 1 \
    --nproc_per_node 1 \
    run_clm_pt_with_peft.py \
    --model_name_or_path meta-llama/Llama-3.2-1B \
    --tokenizer_name_or_path polyglots/Extended-Sinhala-LLaMA \
    --dataset_dir ./sinhala_data \
    --data_cache_dir ./data_cache \
    --validation_split_percentage 0.001 \
    --per_device_train_batch_size 16 \
    --per_device_eval_batch_size 16 \
    --do_train \
    --seed 42 \
    --bf16 \
    --num_train_epochs 1 \
    --lr_scheduler_type cosine \
    --learning_rate 2e-4 \
    --warmup_ratio 0.05 \
    --weight_decay 0.01 \
    --logging_strategy steps \
    --logging_steps 10 \
    --logging_first_step True \
    --save_strategy steps \
    --save_steps 200 \
    --save_total_limit 3 \
    --gradient_accumulation_steps 8 \
    --preprocessing_num_workers 8 \
    --block_size 512 \
    --output_dir ./output_sinllama_1b \
    --overwrite_output_dir \
    --ddp_timeout 30000 \
    --lora_rank 8 \
    --lora_alpha 32 \
    --lora_dropout 0.05 \
    --trainable q_proj,v_proj,k_proj,o_proj,gate_proj,down_proj,up_proj \
    --modules_to_save embed_tokens,lm_head \
    --torch_dtype bfloat16

## 6. Push adapter to HuggingFace (isji)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

adapter_path = './output_sinllama_1b/pt_lora_model'
repo_id = 'isji/sinllama-1b-cpt'

# Load tokenizer first so we know the vocab size
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

config = PeftConfig.from_pretrained(adapter_path)
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    dtype=torch.float16,
)
# Base model has 128,256 tokens; extended Sinhala tokenizer has 139,336 — resize before loading adapter
model.resize_token_embeddings(len(tokenizer))

model = PeftModel.from_pretrained(model, adapter_path)

model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print(f'Adapter pushed to https://huggingface.co/{repo_id}')

## 7. Quick Sinhala inference test

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

adapter_path = './output_sinllama_1b/pt_lora_model'

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
config = PeftConfig.from_pretrained(adapter_path)
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    dtype=torch.float16,
    device_map='auto',
)
model.resize_token_embeddings(len(tokenizer))
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

def generate(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# Test prompts — give it a Sinhala start and see if it continues naturally
prompts = [
    "ශ්‍රී ලංකාව",          # Sri Lanka
    "බුද්ධ ධර්මය",           # Buddhist teaching
    "සිංහල භාෂාව",          # Sinhala language
]

for p in prompts:
    print(f"Prompt : {p}")
    print(f"Output : {generate(p)}")
    print("-" * 60)

## 8. Same test with base Llama-3.2-1B (no Sinhala training)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

base_model_id = 'meta-llama/Llama-3.2-1B'

base_tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    dtype=torch.float16,
    device_map='auto',
)
base_model.eval()

def generate_base(prompt, max_new_tokens=100):
    inputs = base_tokenizer(prompt, return_tensors='pt').to(base_model.device)
    with torch.no_grad():
        out = base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
        )
    return base_tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

prompts = [
    "ශ්‍රී ලංකාව",
    "බුද්ධ ධර්මය",
    "සිංහල භාෂාව",
]

for p in prompts:
    print(f"Prompt : {p}")
    print(f"Output : {generate_base(p)}")
    print("-" * 60)